In [1]:
# The line below allows to print all the outputs of a cell instead of only the last one
# %config InteractiveShell.ast_node_interactivity = "all"
import os
import pathlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

In [3]:
from_script = True

print(f"WARNING: This notebook is set with from_script = {from_script}.")

if from_script:
    # Get the path of the notebook config file from the environment variable
    path_config_notebook = os.environ["PATH_YAML_CONFIG"]
    # Load the notebook config file
    with open(path_config_notebook, "r") as file:
        dict_config_notebook = yaml.safe_load(file)
        print("The dict_config_notebook is:")
        print(dict_config_notebook)
    ID_XP_DATASET = dict_config_notebook["id_xp_dataset"]
    ID_XP = dict_config_notebook["id_xp"]
    PATH_PROJECT = pathlib.Path(dict_config_notebook["path_project"])
    DICT_FILTERS = dict_config_notebook.get("dict_filters", None)
    NAME_SEABORN_COLUMN = dict_config_notebook.get("name_seaborn_column", "delay_obs")
    NAME_SEABORN_ROW = dict_config_notebook.get("name_seaborn_row", "env_name")
    NAME_SEABORN_HUE = dict_config_notebook.get("name_seaborn_hue", "trainer_name")
    NAME_COLUMN_TO_LOOP_OVER = dict_config_notebook.get(
        "column_to_loop_over", "mlflow_experiment_name"
    )
    NAME_MLFLOW_STORE = dict_config_notebook.get(
        "name_mlflow_store", "mlruns_ruche_downloaded"
    )
else:
    ID_XP_DATASET = 21
    ID_XP = 24
    PATH_PROJECT = pathlib.Path(
        "/home/hosseinkhan/Documents/work/phd/git_repositories/control_dde"
    )
    DICT_FILTERS = {
        "env_name": ["cylinder"],
        "env_params_max_control": [0.0],
        "env_params_reynolds": [120.0, 90.0],
    }

    # DICT_FILTERS = None
    NAME_SEABORN_COLUMN = "delay_obs"
    NAME_SEABORN_ROW = "env_name"
    NAME_SEABORN_HUE = "trainer_name"
    NAME_COLUMN_TO_LOOP_OVER = "env_params_reynolds"
    NAME_MLFLOW_STORE = "mlruns_ruche_downloaded"

# Add the path of the project to the sys.path in order to import
# the modules in the src folder
sys.path.insert(0, os.path.abspath(PATH_PROJECT))

# Define the paths
path_mlruns = pathlib.Path(f"{PATH_PROJECT}/data/{NAME_MLFLOW_STORE}")
path_experiment_dataset = pathlib.Path(f"{path_mlruns}/{ID_XP_DATASET}")
path_experiment = pathlib.Path(f"{path_mlruns}/{ID_XP}")
# Get all folder in the mlruns/id_xp directory except the folder 'tags'
# using pathlib and iter dir
list_id_hash = [
    path.name
    for path in path_experiment.iterdir()
    if path.is_dir() and path.name != "tags"
]
list_id_hash_dataset = [
    path.name
    for path in path_experiment_dataset.iterdir()
    if path.is_dir() and path.name != "tags"
]

print(f"Number of runs in the dataset: {len(list_id_hash_dataset)} \n")
print(f"Number of runs: {len(list_id_hash)} \n")
print(f"XP id dataset: {ID_XP_DATASET} \n")
print(f"XP id: {ID_XP} \n")

nested_dict_config_dataset = {}
nested_dict_config = {}
list_df_config_flattened_dataset = []
list_df_config_flattened = []


def check_xp_lifecycle(path_file: pathlib.Path):
    if not path_file.exists():
        raise FileNotFoundError(f"The file {path_file} does not exist.")
    if not path_file.is_file():
        raise FileNotFoundError(f"The path {path_file} is not a file.")
    # Load the yaml file
    with open(path_file, "r") as _file:
        dict_temp = yaml.safe_load(_file)
    # Check the lifecycle key
    assert dict_temp.get("lifecycle_stage") == "active", "The experiment is not active."


# Verify all experiments are done on the same environment:
path_xp = f"{path_mlruns}/{ID_XP}"
for name_id in list_id_hash:
    # Check file lifecycle
    list_glob_xp_metadata = list(pathlib.Path(f"{path_xp}/{name_id}").glob("meta.yaml"))
    assert len(list_glob_xp_metadata) != 0, "No metadata file in the directory."
    assert (
        len(list_glob_xp_metadata) == 1
    ), "More than one metadata file in the directory."
    check_xp_lifecycle(list_glob_xp_metadata[0])

    # Get the config file
    list_glob_config = list(
        pathlib.Path(f"{path_xp}/{name_id}/artifacts").glob("./config/*.yaml")
    )
    assert len(list_glob_config) != 0, "No config file in the directory."
    assert len(list_glob_config) == 1, "More than one config file in the directory."

    name_config_file = list_glob_config[0].name
    # Check if the file is a yaml file
    assert name_config_file.endswith(".yaml"), "File is not a yaml file."

    path_run_config_yaml = list_glob_config[0]
    with open(path_run_config_yaml, "r") as file:
        # Get the config file as a dictionary
        dict_config_temp = yaml.safe_load(file)
        # Add the config file to the nested dictionary
        nested_dict_config[name_id] = dict_config_temp
        # Flatten the config file for easier check
        list_df_config_flattened.append(pd.json_normalize(dict_config_temp, sep="_"))
    del dict_config_temp

path_xp_dataset = f"{path_mlruns}/{ID_XP_DATASET}"
for name_id in list_id_hash_dataset:
    list_glob_config = list(
        pathlib.Path(f"{path_xp_dataset}/{name_id}/generated_data").glob(
            "./config/*.yaml"
        )
    )
    assert len(list_glob_config) != 0, "No config file in the directory."
    assert len(list_glob_config) == 1, "More than one config file in the directory."

    name_config_file = list_glob_config[0].name
    # Check if the file is a yaml file
    assert name_config_file.endswith(".yaml"), "File is not a yaml file."

    path_run_config_yaml = list_glob_config[0]
    with open(path_run_config_yaml, "r") as file:
        # Get the config file as a dictionary
        dict_config_temp = yaml.safe_load(file)
        # Add the config file to the nested dictionary
        nested_dict_config_dataset[name_id] = dict_config_temp
        # Flatten the config file for easier check
        list_df_config_flattened_dataset.append(
            pd.json_normalize(dict_config_temp, sep="_")
        )
    del dict_config_temp

KeyError: 'PATH_YAML_CONFIG'

## DataFrames generation

### DataFrame: Training config

In [ ]:
df_trainining_config = (
    pd.concat(
        [
            df_config_flattened.assign(training_id=name_id)
            for name_id, df_config_flattened in zip(
                list_id_hash, list_df_config_flattened
            )
        ]
    )
    .set_index("training_id")
    .pipe(
        lambda df: df.astype(
            {col: "float" for col in df.select_dtypes(include="int").columns}
        )
    )
)

df_trainining_config.head(10)

### DataFrame: Dataset config

In [ ]:
df_dataset_config = (
    pd.concat(
        [
            df_config_flattened.assign(dataset_id=name_id)
            for name_id, df_config_flattened in zip(
                list_id_hash_dataset, list_df_config_flattened_dataset
            )
        ]
    )
    .set_index("dataset_id")
    .pipe(
        lambda df: df.astype(
            {col: "float" for col in df.select_dtypes(include="int").columns}
        )
    )
)

df_dataset_config.head(10)

In [5]:
print("Unique values of the dataset DataFrame")
print(df_dataset_config.nunique())
for column in df_dataset_config.columns:
    print(f"{column}: {df_dataset_config[column].unique()}")

Unique values of the dataset DataFrame
mlflow_experiment_name                   1
seed                                     1
data_control_type                        1
data_num_steps                           1
data_num_trajectories                    2
data_type                                1
env_name                                 3
env_params_delay_observation             1
env_params_dict_solver_dt                2
env_params_dict_solver_name              1
env_params_dict_solver_order             1
env_params_dict_solver_stabilization     1
env_params_dt                            1
env_params_log_callback_interval         2
env_params_max_control                   2
env_params_mesh                          1
env_params_name_flow                     3
env_params_paraview_callback_interval    2
env_params_reynolds                      7
dtype: int64
mlflow_experiment_name: ['2024_11_25_training_data_20h_58m']
seed: [0.]
data_control_type: ['uniform']
data_num_steps: [200.]
data_

### DataFrame: Training and dataset config merged

#### Filtering DataFrame

In [6]:
# Filtering function based on the DICT_FILTERS
def filter_dataframe(df: pd.DataFrame):
    mask = pd.Series([True] * len(df), index=df.index)
    if DICT_FILTERS is None:
        return mask
    else:
        for key, list_values in DICT_FILTERS.items():
            if key in df.columns:
                mask = mask & df[key].isin(list_values)
            else:
                print(f"Warning: The key {key} is not in the config file.")
    return mask


df_training_dataset_merged = (
    df_trainining_config.assign(
        dataset_id=lambda x: x.data_path_data_folder.str.extract(
            rf"{ID_XP_DATASET}/(.*?)/generated_data"
        )
    )
    .join(df_dataset_config, on="dataset_id", rsuffix="_dataset")
    .fillna({"model_params_n_delays": 0.0})
    .drop(columns=["data_path_data_folder"])
    .loc[lambda df: filter_dataframe(df)]
)

In [7]:
df_training_dataset_merged.head(10)

,mlflow_experiment_name,seed,data_normaliser_name,model_name,model_params_activation,model_params_delay_exponential_dist_param,model_params_depth,model_params_dropout,model_params_final_activation,model_params_width_size,...,env_params_dict_solver_name,env_params_dict_solver_order,env_params_dict_solver_stabilization,env_params_dt,env_params_log_callback_interval,env_params_max_control,env_params_mesh,env_params_name_flow,env_params_paraview_callback_interval,env_params_reynolds
training_id,,,,,,,,,,,,,,,,,,,,,
830c48fa6f874d9989a8d032c7e9009f,2024_11_27_training_00h_04m,0.0,standard_normal,ndde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,120.0
c26dd64706334340a840f421d7a995dc,2024_11_27_training_00h_04m,1.0,standard_normal,node,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
6e12ee4af8c5492487a35adaed03d876,2024_11_27_training_00h_04m,1.0,standard_normal,ncdde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
47ad805accce4519af72581a379435b4,2024_11_27_training_00h_04m,0.0,standard_normal,ncde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,120.0
3b17acdf94104d1a86a515124c4c32ed,2024_11_27_training_00h_04m,1.0,standard_normal,ncde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,120.0
ca2c6e16ca264ba19663f35c10323c5d,2024_11_27_training_00h_04m,1.0,standard_normal,node,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,120.0
fa3979d35849421db77b27d4207b4a1b,2024_11_27_training_00h_04m,0.0,standard_normal,node,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,120.0
863670e175e44d17b359f72de1bf363b,2024_11_27_training_00h_04m,1.0,standard_normal,ndde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
f85f98d5d83843e9b0c9ac49af73dbdb,2024_11_27_training_00h_04m,0.0,standard_normal,ncde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0


In [8]:
df_training_dataset_merged.shape

(16, 40)

### DataFrame: Training metrics

In [9]:
# noinspection PyUnresolvedReferences
import tbparse

In [ ]:
list_df_run_tb_data = []
list_id_hash_filtered = list(df_training_dataset_merged.index)

for name_id in list_id_hash_filtered:
    path_run_config_folder = (
        f"{path_xp}/{name_id}/generated_data/trainer_data/ode_trainer"
    )
    # Get the files which have "tfevents" in their name
    list_files = [
        path.name for path in pathlib.Path(path_run_config_folder).glob("*tfevents*")
    ]
    assert len(list_files) == 1, f"More than one file in {path_run_config_folder}"
    path_run_config_file = f"{path_run_config_folder}/{list_files[0]}"
    # Load config with tbparser
    # noinspection PyPackageRequirements

    tb_reader = tbparse.SummaryReader(path_run_config_file)
    df_run_tb_data = tb_reader.scalars
    list_df_run_tb_data.append(df_run_tb_data)

In [11]:
df_tb_data = pd.concat(
    list_df_run_tb_data,
    # axis="columns",
    keys=list_id_hash_filtered,
    names=["run_id", "episode_data"],
).rename(columns={"tag": "metric"})
df_tb_data.head()

step           metric  \
run_id                           episode_data                          
830c48fa6f874d9989a8d032c7e9009f 0               50  magnitude/truth   
                                 1               50  magnitude/truth   
                                 2               50  magnitude/truth   
                                 3               50  magnitude/truth   
                                 4              100  magnitude/truth   

                                                   value  
run_id                           episode_data             
830c48fa6f874d9989a8d032c7e9009f 0             10.459950  
                                 1             14.187618  
                                 2              9.589944  
                                 3             17.069220  
                                 4             16.542973

In [12]:
df_tb_data.metric.unique()

array(['magnitude/truth', 'mean_loss_over_batches/eval',
       'mean_loss_over_batches/train'], dtype=object)

In [13]:
df_training_dataset_merged.sample(frac=1).head(10)

,mlflow_experiment_name,seed,data_normaliser_name,model_name,model_params_activation,model_params_delay_exponential_dist_param,model_params_depth,model_params_dropout,model_params_final_activation,model_params_width_size,...,env_params_dict_solver_name,env_params_dict_solver_order,env_params_dict_solver_stabilization,env_params_dt,env_params_log_callback_interval,env_params_max_control,env_params_mesh,env_params_name_flow,env_params_paraview_callback_interval,env_params_reynolds
training_id,,,,,,,,,,,,,,,,,,,,,
d2564c5c6d54446d9e3ad04886d71210,2024_11_27_training_00h_04m,1.0,standard_normal,ndde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,120.0
fdcc15726c764b70a08f5400d4264a45,2024_11_27_training_00h_04m,0.0,standard_normal,ndde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
2d8808d002604e068461f64998ca2c4c,2024_11_27_training_00h_04m,0.0,standard_normal,node,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
6e12ee4af8c5492487a35adaed03d876,2024_11_27_training_00h_04m,1.0,standard_normal,ncdde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
f85f98d5d83843e9b0c9ac49af73dbdb,2024_11_27_training_00h_04m,0.0,standard_normal,ncde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
b33cb60922c84370a5c6fca47da8dee6,2024_11_27_training_00h_04m,1.0,standard_normal,ncde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
bed31abf3f0d4498879e8c3d617b93d3,2024_11_27_training_00h_04m,1.0,standard_normal,ncdde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,120.0
863670e175e44d17b359f72de1bf363b,2024_11_27_training_00h_04m,1.0,standard_normal,ndde,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0
c26dd64706334340a840f421d7a995dc,2024_11_27_training_00h_04m,1.0,standard_normal,node,relu,5.0,2.0,0.0,identity,64.0,...,semi_implicit_bdf,2.0,none,0.01,100.0,0.0,coarse,cylinder,100.0,90.0


In [14]:
df_all_data = (
    df_tb_data.join(df_training_dataset_merged, on="run_id")
    # .set_index(df_training_dataset_merged.columns.tolist(), append=True)
)

In [15]:
df_all_data.loc[df_all_data.metric.str.contains("loss")].metric.unique()

array(['mean_loss_over_batches/eval', 'mean_loss_over_batches/train'],
      dtype=object)

In [ ]:
list_unique_values_column_to_loop_over = df_all_data[NAME_COLUMN_TO_LOOP_OVER].unique()

for value in list_unique_values_column_to_loop_over:
    print(f"Column values to loop over: {value}")

In [ ]:
# Plot a seaborn relplot
sns.set_theme(style="whitegrid")

list_unique_values_column_to_loop_over = df_all_data[NAME_COLUMN_TO_LOOP_OVER].unique()


for column_value in list_unique_values_column_to_loop_over:
    print(f"COLUMN VALUE: {column_value}")

    df_sns = (
        df_all_data.loc[df_all_data[NAME_COLUMN_TO_LOOP_OVER] == column_value]
        .loc[df_all_data.metric.str.contains("loss")]
        .rename(columns={"env_params_delay_observation": "delay_obs"})
    )

    color_palette = {
        "node": "forestgreen",
        "ncde": "limegreen",
        "ndde": "blue",
        "ncdde": "skyblue",
    }

    g = sns.relplot(
        data=df_sns,
        x="step",
        y="value",
        style="metric",
        kind="line",
        hue=NAME_SEABORN_HUE,
        row=NAME_SEABORN_ROW,
        # col="delay_obs",
        col=NAME_SEABORN_COLUMN,
        # no confidence interval
        errorbar=None,
        # No sharing of the y axis
        facet_kws={"sharey": False},
        palette=color_palette,
    )
    # Y-scale is logarithmic
    g.set(yscale="log")
    plt.show()